# Tutorial - VaR Paramétrico de Portafolio
Sergio Cabrales, Universidad de los Andes

https://www.sac2.com/

## 1. Carga de librerías, funciones y APIs necesarias.

#### 1.1. Instalan las librerías que no incluye Google Colab

In [ ]:
pip install yfinance

In [ ]:
pip install mplfinance

In [ ]:
pip install arch

#### 1.2. Se cargan las librerías requeridas

In [ ]:
# Funciones numéricas adicionales
import numpy as np

# Lectura de datos y manejo de Data-sets
import pandas as pd

# Datos
import yfinance as yfin

# Gráficos
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm
import seaborn as sns

# Análisis técnico
import mplfinance as mpf

# Probabilidad y estadística
import math
from scipy.stats import norm, chi2, jarque_bera, shapiro
from scipy.optimize import brentq
from scipy import stats

# ARCH model
from arch import arch_model

# Engle test for ARCH effects
from statsmodels.stats.diagnostic import het_arch

## 2. Definición del Portafolio y Obtención de Datos

#### 2.1. Definición del portafolio de acciones del sector energético

In [ ]:
# Portafolio de acciones del sector de Oil & Gas
tickers = {
    # Majors (integradas)
    "XOM":  "ExxonMobil",
    "CVX":  "Chevron",
    "SHEL": "Shell",
    "BP":   "BP",
    "TTE":  "TotalEnergies",
    "EQNR": "Equinor",

    # Latinoamérica / NOCs listadas
    "EC":   "Ecopetrol",
    "PBR":  "Petrobras",

    # Upstream (E&P)
    "COP":  "ConocoPhillips",
    "EOG":  "EOG Resources",
    "OXY":  "Occidental Petroleum",
    "HES":  "Hess",
    "MRO":  "Marathon Oil",
    "FANG": "Diamondback Energy",

    # Refinación y midstream
    "MPC":  "Marathon Petroleum",
    "PSX":  "Phillips 66",
    "VLO":  "Valero Energy",
}

ticker_list = list(tickers.keys())
n = len(ticker_list)

# Ponderaciones del portafolio (equal-weighted por defecto)
weights = np.array([1/n] * n)

print(f"Portafolio con {n} activos (ponderación igual: {100/n:.2f}% cada uno)")
for t, w in zip(ticker_list, weights):
    print(f"  {t:5s}  {tickers[t]:<25s}  peso: {w:.4f}")

#### 2.2. Descarga de datos históricos desde Yahoo Finance

https://finance.yahoo.com/

In [ ]:
start_date = "2020-01-01"

# Descarga de precios de cierre ajustados para todos los tickers
raw = yfin.download(ticker_list, start=start_date, multi_level_index=True)
prices = raw["Close"]

# Drop columns (tickers) that contain any NaN values
initial_columns = prices.columns.tolist()
prices = prices.dropna(axis=1)
cleaned_columns = prices.columns.tolist()

# Update ticker_list to only include tickers with complete data
ticker_list = cleaned_columns
n = len(ticker_list)
weights = np.array([1/n] * n)

# Drop rows that might have NaN after column drop if any
prices = prices.dropna(how="all")

print(f"Datos descargados: {prices.shape[0]} observaciones, {prices.shape[1]} activos")
print(f"Tickers removidos debido a datos faltantes: {list(set(initial_columns) - set(cleaned_columns))}")
print(f"Período: {prices.index[0].date()} → {prices.index[-1].date()}")
prices.tail()

## 3. Visualización de Precios del Portafolio

### 3.1. Evolución de precios normalizados (base 100)

In [ ]:
# Normalizar precios a base 100 para comparar evolución
prices_norm = prices / prices.iloc[0] * 100

fig, ax = plt.subplots(figsize=(16, 8))
for ticker in ticker_list:
    ax.plot(prices_norm.index, prices_norm[ticker], linewidth=1.2, label=f"{ticker} — {tickers[ticker]}")

ax.axhline(100, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_title('Precios Normalizados (Base 100) — Portafolio Oil & Gas', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Precio Normalizado')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.set_facecolor('#f8f9fa')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2. Mapa de calor de correlaciones

In [ ]:
# Retornos individuales para correlación
# Ensure prices used for correlation are clean of NaNs by dropping columns with NaNs
prices_for_corr = prices.dropna(axis=1)
log_ret_all = np.log(prices_for_corr / prices_for_corr.shift(1)).dropna()

# If after cleaning, log_ret_all is empty or has too few observations, inform the user.
if log_ret_all.empty or log_ret_all.shape[1] < 2:
    print("Warning: log_ret_all is empty or has too few columns after dropping NaN columns and rows. Correlation matrix cannot be computed.")
    # Display an empty plot or a message if no data is available
    fig, ax = plt.subplots(figsize=(12, 10))
    ax.text(0.5, 0.5, "No data to compute correlation matrix after cleaning.",
            horizontalalignment='center', verticalalignment='center', transform=ax.transAxes, fontsize=12)
    ax.set_title('Matriz de Correlaciones — Retornos Log Diarios', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(12, 10))
    corr_matrix = log_ret_all.corr()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
                cmap="RdYlGn", center=0, vmin=-1, vmax=1,
                linewidths=0.5, ax=ax, annot_kws={"size": 8})
    ax.set_title('Matriz de Correlaciones — Retornos Log Diarios', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Retornos del Portafolio

### 4.1. Retornos Logarítmicos Individuales

Los retornos logarítmicos de cada activo $ se calculan como:
13952
r_{i,t} = \ln\left(\frac{S_{i,t}}{S_{i,t-1}}\right)
13952

In [ ]:
# Retornos logarítmicos individuales
log_ret_all = np.log(prices / prices.shift(1))
# Replace inf/-inf with NaN and then drop NaNs to ensure finite values only
log_ret_all = log_ret_all.replace([np.inf, -np.inf], np.nan).dropna()
print("Retornos logarítmicos individuales:")
log_ret_all.tail()

### 4.2. Retornos del Portafolio

El retorno del portafolio en cada período es la suma ponderada de los retornos individuales:
\
r_{p,t} = \sum_{i=1}^{N} w_i \cdot r_{i,t}


In [ ]:
# Retorno del portafolio como suma ponderada
df = pd.DataFrame(index=log_ret_all.index) # Use the index of the cleaned log_ret_all
df['Log Returns'] = log_ret_all.dot(weights)
print("Retorno anualizado del portafolio:", round(df['Log Returns'].mean() * 252 * 100, 4), "%")
df[['Log Returns']].tail()

### 4.3. Retornos Logarítmicos Anualizados

13952r_p = 252 \cdot \bar{r}_{p,t}13952

In [ ]:
LogReturns = np.mean(df["Log Returns"]) * 252
print("Retorno log anual del portafolio: {:.4f} %".format(100 * LogReturns))

### 4.4. Gráfica de Retornos del Portafolio

In [ ]:
# Gráfico simple de retornos
plt.figure(figsize=(15, 8))
plt.plot(df['Log Returns'], color='blue', linewidth=0.8)
plt.title('Retornos Logarítmicos del Portafolio Oil & Gas')
plt.xlabel('Fecha')
plt.ylabel('Retorno Log')
plt.show()

In [ ]:
import matplotlib.patches as mpatches

fig, axes = plt.subplots(3, 1, figsize=(15, 12), gridspec_kw={'height_ratios': [3, 1.5, 1.5]})
fig.suptitle('Análisis de Retornos Logarítmicos — Portafolio Oil & Gas', fontsize=16, fontweight='bold', y=0.98)

ax1 = axes[0]
returns = df['Log Returns'].dropna()

ax1.bar(returns.index, returns.where(returns >= 0), color='#2ecc71', alpha=0.7, width=1, label='Positivo')
ax1.bar(returns.index, returns.where(returns < 0),  color='#e74c3c', alpha=0.7, width=1, label='Negativo')
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.axhline(returns.mean(), color='navy', linewidth=1.2, linestyle='--', label=f'Media: {returns.mean():.4f}')
ax1.axhline(returns.mean() + 2*returns.std(), color='orange', linewidth=1, linestyle=':', label=f'±2σ: {2*returns.std():.4f}')
ax1.axhline(returns.mean() - 2*returns.std(), color='orange', linewidth=1, linestyle=':')
ax1.set_title('Retornos Logarítmicos Diarios del Portafolio', fontsize=13)
ax1.set_ylabel('Retorno Log')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_facecolor('#f8f9fa')
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
cumulative = returns.cumsum()
ax2.plot(cumulative.index, cumulative, color='#2c3e50', linewidth=1.5)
ax2.fill_between(cumulative.index, cumulative, 0, where=(cumulative >= 0), color='#2ecc71', alpha=0.3)
ax2.fill_between(cumulative.index, cumulative, 0, where=(cumulative < 0),  color='#e74c3c', alpha=0.3)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_title('Retorno Logarítmico Acumulado del Portafolio', fontsize=13)
ax2.set_ylabel('Retorno Acumulado')
ax2.set_facecolor('#f8f9fa')
ax2.grid(True, alpha=0.3)

ax3 = axes[2]
ax3.hist(returns, bins=60, density=True, color='steelblue', alpha=0.6, edgecolor='white')
mu, sigma = returns.mean(), returns.std()
x = np.linspace(returns.min(), returns.max(), 300)
ax3.plot(x, norm.pdf(x, mu, sigma), 'r-', linewidth=2, label=f'Normal(μ={mu:.4f}, σ={sigma:.4f})')
ax3.axvline(mu, color='navy', linestyle='--', linewidth=1.2)
ax3.set_title('Distribución de Retornos del Portafolio', fontsize=13)
ax3.set_xlabel('Retorno Log')
ax3.set_ylabel('Densidad')
ax3.legend(fontsize=9)
ax3.set_facecolor('#f8f9fa')
ax3.grid(True, alpha=0.3)

stats_text = (f"Obs: {len(returns):,}  |  Media: {mu:.4f}  |  σ: {sigma:.4f}  |  "
              f"Skew: {returns.skew():.3f}  |  Kurt: {returns.kurtosis():.3f}  |  "
              f"Min: {returns.min():.4f}  |  Max: {returns.max():.4f}")
fig.text(0.5, 0.01, stats_text, ha='center', fontsize=9, color='#555',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#ecf0f1', alpha=0.8))
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

## 5. Volatilidad del Portafolio

### 5.1 Volatilidad diaria y anual

La volatilidad diaria es la desviación estándar de los retornos del portafolio:

13952\sigma_p = \text{desv}(r_p) = \sqrt{\text{Var}(r_p)}13952

La volatilidad anualizada:
13952\sigma_p^{anual} = \sigma_p \cdot \sqrt{252}13952

In [ ]:
vol_d = np.std(df['Log Returns'])
vol_a = vol_d * np.sqrt(252)

print("Volatilidad diaria del portafolio:    {:.4f} %".format(100 * vol_d))
print("Volatilidad anualizada del portafolio: {:.4f} %".format(100 * vol_a))

# Comparación de volatilidades individuales
ind_vols = log_ret_all.std() * np.sqrt(252) * 100
port_vol = vol_a * 100

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(ticker_list, ind_vols, color='steelblue', alpha=0.7, label='Volatilidad individual')
ax.axhline(port_vol, color='red', linewidth=2, linestyle='--', label=f'Portafolio: {port_vol:.2f}%')
ax.set_title('Volatilidad Anualizada: Activos Individuales vs. Portafolio', fontsize=13, fontweight='bold')
ax.set_xlabel('Ticker')
ax.set_ylabel('Volatilidad Anualizada (%)')
ax.legend(fontsize=10)
ax.set_facecolor('#f8f9fa')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, ind_vols):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"{val:.1f}%",
            ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Comparación de volatilidades individuales
ind_vols = log_ret_all.std() * np.sqrt(252) * 100
port_vol = vol_a * 100

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(ticker_list, ind_vols, color='steelblue', alpha=0.7, label='Volatilidad individual')
ax.axhline(port_vol, color='red', linewidth=2, linestyle='--', label=f'Portafolio: {port_vol:.2f}%')
ax.set_title('Volatilidad Anualizada: Activos Individuales vs. Portafolio', fontsize=13, fontweight='bold')
ax.set_xlabel('Ticker')
ax.set_ylabel('Volatilidad Anualizada (%)')
ax.legend(fontsize=10)
ax.set_facecolor('#f8f9fa')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, ind_vols):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"{val:.1f}%",
            ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

## 6. Normalidad de los Retornos del Portafolio

### 6.1. Histograma

In [ ]:
k = int(math.sqrt(len(df['Log Returns'])))
plt.hist(df['Log Returns'], bins=k)
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.title('Histogram of Portfolio Log-Returns')
plt.show()

### 6.2. Q-Q Plot

In [ ]:
mean  = np.mean(df['Log Returns'])
std_dev = np.std(df['Log Returns'])
sm.qqplot(df['Log Returns'], stats.norm, loc=mean, scale=std_dev, line='45')
plt.title("Q-Q Plot — Retornos del Portafolio")
plt.show()

### 6.3. P-P Plot

In [ ]:
mean    = np.mean(df['Log Returns'])
std_dev = np.std(df['Log Returns'])
n_obs   = len(df['Log Returns'])
fig, ax = plt.subplots()
p  = np.arange(1, n_obs + 1) / n_obs - 0.5 / n_obs
pp = np.sort(stats.norm.cdf(df['Log Returns'], loc=mean, scale=std_dev))
sns.scatterplot(x=pp, y=p, color='blue', edgecolor='blue', ax=ax)
ax.set_title('P-P Plot — Retornos del Portafolio')
ax.set_xlabel('Theoretical Probabilities')
ax.set_ylabel('Sample Probabilities')
ax.margins(x=0, y=0)
plt.plot(np.linspace(0, 1.01), np.linspace(0, 1.01), 'r', lw=2)
plt.show()

### 6.4. Jarque-Bera Test

In [ ]:
jb_value, p_value = jarque_bera(df['Log Returns'])
print("Jarque-Bera value: ", jb_value)
print("p-value: ", p_value)

if p_value > 0.05:
    print("The data is normally distributed")
else:
    print("The data is not normally distributed")

## 7. Autocorrelación

### 7.1. Autocorrelation

In [ ]:
plot_acf(df['Log Returns'], lags=21)
plt.show()

In [ ]:
df['Log Returns^2'] = df['Log Returns'] ** 2

In [ ]:
plot_acf(df['Log Returns^2'], lags=21)
plt.show()

### 7.2. Partial Autocorrelation

In [ ]:
plot_pacf(df['Log Returns'], lags=21)
plt.show()

In [ ]:
plot_pacf(df['Log Returns^2'], lags=21)
plt.show()

### 7.3. Ljung-Box Test

In [ ]:
lags = 5
lb_test = sm.stats.diagnostic.acorr_ljungbox(df['Log Returns'], lags)
print("Ljung-Box value: ", lb_test.iloc[-1, 0])
print("p-value: ", lb_test.iloc[-1, 1])

if lb_test.iloc[-1, 1] > 0.05:
    print("The data is independent")
else:
    print("The data is not independent")

## 8. Efectos ARCH

### 8.1. Ljung-Box Test — Retornos al Cuadrado

In [ ]:
lags = 6
lb_test = sm.stats.diagnostic.acorr_ljungbox(df['Log Returns^2'], lags)
print("Ljung-Box value: ", lb_test.iloc[-1, 0])
print("p-value: ", lb_test.iloc[-1, 1])

if lb_test.iloc[-1, 1] > 0.05:
    print("The data is independent")
else:
    print("The data is not independent")

## 9. Modelo GARCH sobre el Portafolio

### 9.1. GARCH(1,1)

13952\sigma_{t}^{2} = \alpha_0 + \alpha_1 \epsilon_{t-1}^{2} + \beta_1 \sigma_{t-1}^{2}13952

Se ajusta el modelo GARCH(1,1) directamente sobre los retornos del portafolio.

In [ ]:
model = arch_model(df['Log Returns'], mean='Zero', vol='GARCH', q=1, p=1, rescale=False)
results = model.fit()
print(results.summary())

In [ ]:
params = pd.DataFrame(results.params)
params

In [ ]:
e = pd.DataFrame(results.std_resid)
e['resid^2'] = e['std_resid'] ** 2
e

### 9.2. Histograma de Residuos

In [ ]:
k = int(math.sqrt(len(e['std_resid'])))
plt.hist(e['std_resid'], bins=k)
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.title('Histogram of Residuals — GARCH(1,1) Portfolio')
plt.show()

### 9.3. Q-Q Plot — Residuos

In [ ]:
pplot = sm.ProbPlot(e['std_resid'], stats.norm, fit=True)
fig   = pplot.qqplot(line="45")
plt.title("Q-Q Plot — GARCH Residuals")
plt.show()

### 9.4. P-P Plot — Residuos

In [ ]:
pplot = sm.ProbPlot(e['std_resid'], stats.norm, fit=True)
fig   = pplot.ppplot(line="45")
plt.title("P-P Plot — GARCH Residuals")
plt.show()

### 9.5. Autocorrelación — Residuos

In [ ]:
plot_acf(e['std_resid'], lags=21)
plt.show()

In [ ]:
plot_acf(e['resid^2'], lags=21)
plt.show()

### 9.6. Jarque-Bera Test — Residuos

In [ ]:
jb_value, p_value = jarque_bera(e['std_resid'])
print("Jarque-Bera value: ", jb_value)
print("p-value: ", p_value)
if p_value > 0.05:
    print("The data is normally distributed")
else:
    print("The data is not normally distributed")

### 9.7. Ljung-Box Test — Residuos al Cuadrado

In [ ]:
lags = 5
lb_test = sm.stats.diagnostic.acorr_ljungbox(e['resid^2'], lags)
print("Ljung-Box value: ", lb_test.iloc[-1, 0])
print("p-value: ", lb_test.iloc[-1, 1])
if lb_test.iloc[-1, 1] > 0.05:
    print("The data is independent")
else:
    print("The data is not independent")

### 9.8. Volatilidad Condicional — Pronóstico

In [ ]:
fig = results.plot(annualize="D")

In [ ]:
df['Volatility'] = results.conditional_volatility * math.sqrt(252)
plt.figure(figsize=(15, 8))
plt.plot(df['Volatility'], color='orange')
plt.title('Volatilidad Condicional del Portafolio — GARCH(1,1)')
plt.xlabel('Fecha')
plt.ylabel('Volatilidad Anualizada')
plt.show()

In [ ]:
import matplotlib.dates as mdates

fig, axes = plt.subplots(3, 1, figsize=(15, 13), gridspec_kw={'height_ratios': [3, 1.5, 1.5]})
fig.suptitle('Volatilidad Condicional del Portafolio — GARCH(1,1)', fontsize=16, fontweight='bold', y=0.98)

vol = df['Volatility'].dropna()

ax1 = axes[0]
ax1.plot(vol.index, vol, color='#e67e22', linewidth=1.2, label='Vol. Condicional (anualizada)')
p25, p75 = vol.quantile(0.25), vol.quantile(0.75)
p10, p90 = vol.quantile(0.10), vol.quantile(0.90)
mean_vol  = vol.mean()
ax1.axhline(mean_vol, color='navy',  linewidth=1.2, linestyle='--', label=f'Media: {mean_vol:.2%}')
ax1.axhline(p75,      color='gray',  linewidth=0.9, linestyle=':',  label=f'P25-P75: {p25:.2%} – {p75:.2%}')
ax1.axhline(p25,      color='gray',  linewidth=0.9, linestyle=':')
ax1.fill_between(vol.index, p25, p75, alpha=0.12, color='orange', label='Rango intercuartílico')
top5 = vol.nlargest(5)
ax1.scatter(top5.index, top5.values, color='red', zorder=5, s=40, label='Top 5 picos')
for date, value in top5.items():
    ax1.annotate(f"{value:.1%}", xy=(date, value), xytext=(0, 8), textcoords='offset points',
                 fontsize=7.5, color='red', ha='center')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax1.set_ylabel('Volatilidad Anualizada')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_facecolor('#f8f9fa')
ax1.grid(True, alpha=0.3)
ax1.set_title('Volatilidad Condicional Anualizada', fontsize=12)

ax2 = axes[1]
bins   = [0, p25, mean_vol, p75, np.inf]
labels = ['Baja', 'Media-Baja', 'Media-Alta', 'Alta']
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
regimen = pd.cut(vol, bins=bins, labels=labels)
for label, color in zip(labels, colors):
    mask = regimen == label
    ax2.fill_between(vol.index, 0, vol.where(mask), color=color, alpha=0.85, label=label)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax2.set_ylabel('Volatilidad')
ax2.legend(loc='upper left', fontsize=9, ncol=4)
ax2.set_facecolor('#f8f9fa')
ax2.grid(True, alpha=0.3)
ax2.set_title('Régimen de Volatilidad', fontsize=12)

ax3 = axes[2]
ax3.hist(vol, bins=60, density=True, color='#e67e22', alpha=0.65, edgecolor='white')
from scipy.stats import lognorm
shape, loc_ln, scale_ln = lognorm.fit(vol, floc=0)
x = np.linspace(vol.min(), vol.max(), 300)
ax3.plot(x, lognorm.pdf(x, shape, loc_ln, scale_ln), color='darkred', linewidth=2, label='Log-Normal fit')
ax3.axvline(mean_vol, color='navy', linestyle='--', linewidth=1.2, label=f'Media: {mean_vol:.2%}')
ax3.axvline(p25, color='gray', linestyle=':', linewidth=1)
ax3.axvline(p75, color='gray', linestyle=':', linewidth=1, label=f'IQR: [{p25:.2%}, {p75:.2%}]')
ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax3.set_xlabel('Volatilidad Anualizada')
ax3.set_ylabel('Densidad')
ax3.legend(fontsize=9)
ax3.set_facecolor('#f8f9fa')
ax3.grid(True, alpha=0.3)
ax3.set_title('Distribución de la Volatilidad', fontsize=12)

stats_str = (f"Obs: {len(vol):,}  |  Media: {mean_vol:.2%}  |  "
             f"Mín: {vol.min():.2%}  |  Máx: {vol.max():.2%}  |  "
             f"σ(vol): {vol.std():.2%}  |  Actual: {vol.iloc[-1]:.2%}")
fig.text(0.5, 0.005, stats_str, ha='center', fontsize=9, color='#555',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#ecf0f1', alpha=0.8))
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

In [ ]:
print("Último valor de volatilidad del portafolio:", round(df['Volatility'].iloc[-1] * 100, 4), "%")

## 10. VaR Paramétrico del Portafolio

### 10.1. Parámetros

El **VaR Paramétrico Normal** de un día al nivel de confianza $\alpha$ es:
13952
\text{VaR}_{p,t} = \mu_{p,t} + z_{\alpha} \cdot \sigma_{p,t} \cdot \frac{1}{\sqrt{252}}
13952

donde {\alpha} = \Phi^{-1}(1-\alpha)$ y $\sigma_{p,t}$ es la volatilidad condicional GARCH anualizada.

Para el **VaR a 5 días**:
13952
\text{VaR}_{5d} = \sqrt{5} \cdot z_{\alpha} \cdot \sigma_{p,t} \cdot \frac{1}{\sqrt{252}}
13952

In [ ]:
# Nivel de confianza
confidence_level = 0.95

### 10.2. Cálculo del VaR

In [ ]:
# VaR con media deslizante (rolling mean + volatilidad GARCH)
df['VaR'] = None
for i in range(0, len(df)):
    returns = df['Log Returns'].iloc[0:i]
    confidence = confidence_level
    mean_val = returns.mean() if not returns.empty else 0
    df.iloc[i, df.columns.get_loc('VaR')] = (
        mean_val + norm.ppf(1 - confidence) * df.iloc[i, df.columns.get_loc('Volatility')] * np.sqrt(1/252)
    )
df['VaR']

In [ ]:
# VaR sin media (solo cuantil × volatilidad GARCH)
df['VaR_2'] = None
for i in range(0, len(df)):
    df['VaR_2'].iloc[i] = (
        norm.ppf(1 - confidence_level) * df['Volatility'].iloc[i] * np.sqrt(1/252)
    )
df['VaR_2']

In [ ]:
# VaR a 5 días (sqrt(5) × VaR diario)
df['VaR_3'] = None
for i in range(0, len(df)):
    df['VaR_3'].iloc[i] = (
        np.sqrt(5) * norm.ppf(1 - confidence_level) * df['Volatility'].iloc[i] * np.sqrt(1/252)
    )
df['VaR_3']

### 10.3. Gráfico del VaR Paramétrico

In [ ]:
plt.figure(figsize=(15, 8))
plt.plot(df['VaR'], color='red')
plt.title('VaR Paramétrico Normal — Portafolio Oil & Gas (1 día, 99%)')
plt.xlabel('Fecha')
plt.ylabel('VaR')
plt.show()

### 10.4. Gráfico Retorno & VaR

In [ ]:
plt.figure(figsize=(15, 8))
plt.plot(df['Log Returns'],  color='blue',   linewidth=0.8,  label='Retornos Log')
plt.plot(df['VaR'],          color='red',     linewidth=1.2,  label='VaR (media + GARCH) 1d')
plt.plot(df['VaR_2'],        color='orange',  linewidth=1.2,  label='VaR (GARCH) 1d')
plt.plot(df['VaR_3'],        color='cyan',    linewidth=1.2,  label='VaR (GARCH) 5d')
plt.title('Retornos Logarítmicos & VaR Paramétrico Normal — Portafolio Oil & Gas')
plt.xlabel('Fecha')
plt.legend(fontsize=10)
plt.show()

### 10.5. Backtesting: VaR

In [ ]:
# Conteo de excepciones (retornos que superan el VaR en valor absoluto)
df['VaR']   = pd.to_numeric(df['VaR'],   errors='coerce')
df['VaR_2'] = pd.to_numeric(df['VaR_2'], errors='coerce')

violations     = (df['Log Returns'] < df['VaR']).sum()
violations_2   = (df['Log Returns'] < df['VaR_2']).sum()
total_obs      = df['Log Returns'].dropna().shape[0]

print(f"=== Backtesting VaR Paramétrico ({int(confidence_level*100)}%) ===")
print(f"Observaciones totales : {total_obs:,}")
print(f"Violaciones VaR 1 (media+GARCH): {violations} ({100*violations/total_obs:.2f}%)  — esperado: {100*(1-confidence_level):.2f}%")
print(f"Violaciones VaR 2 (GARCH):       {violations_2} ({100*violations_2/total_obs:.2f}%)  — esperado: {100*(1-confidence_level):.2f}%")

In [ ]:
# Gráfico de backtesting
fig, ax = plt.subplots(figsize=(15, 7))
ax.plot(df['Log Returns'], color='#2c3e50', linewidth=0.8, alpha=0.8, label='Retornos Log')
ax.plot(df['VaR_2'],       color='red',    linewidth=1.3, label=f'VaR GARCH 1d ({int(confidence_level*100)}%)')
ax.plot(df['VaR_3'],       color='darkorange', linewidth=1.3, linestyle='--', label=f'VaR GARCH 5d ({int(confidence_level*100)}%)')

# Marcar las violaciones
mask_viol = df['Log Returns'] < df['VaR_2']
ax.scatter(df.index[mask_viol], df['Log Returns'][mask_viol],
           color='red', s=20, zorder=5, label=f'Soprepasadas: {mask_viol.sum()}')

ax.set_title(f'Backtesting VaR Paramétrico — Portafolio Oil & Gas ({int(confidence_level*100)}%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Retorno Log')
ax.legend(fontsize=10)
ax.set_facecolor('#f8f9fa')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 10.6. VaR Actual del Portafolio

In [ ]:
print(f"VaR Paramétrico 1d (media+GARCH)  [{int(confidence_level*100)}%]: {float(df['VaR'].iloc[-1])*100:.4f} %")
print(f"VaR Paramétrico 1d (GARCH)        [{int(confidence_level*100)}%]: {float(df['VaR_2'].iloc[-1])*100:.4f} %")
print(f"VaR Paramétrico 5d (GARCH)        [{int(confidence_level*100)}%]: {float(df['VaR_3'].iloc[-1])*100:.4f} %")